# PHYS 3810: Grover search, one reflection at a time

**Goal:** explain how a phase oracle and a reflection turn a uniformly prepared state into a state more likely to reveal a marked answer. You will predict probabilities, inspect exact states, and sample a small circuit locally.

**Before class:** complete [setup/preflight](00_setup_preflight.ipynb), select **Python (PHYS 3810)**, then choose **Kernel → Restart Kernel and Run All Cells** to verify the environment. No IBM account, credentials, GPU, or internet after installation is needed.

**During class:** restart the kernel and clear all outputs. Read and discuss each prediction before executing its following cell, so the results do not reveal the answers first.

**50-minute core:** setup and notation (5 minutes); oracle (10); diffusion (10); single-target lab (20); exit questions (5). **75-minute expansion:** add marked-set experiments, including the short input-validation check (15), and overshooting (10). All code cells, including the expansion checks, run during validation; the extra discussion/exercises are optional.

Prerequisites: complex amplitudes, probabilities as squared magnitudes, and Hadamard gates. The source is [the reusable Grover module](../../../src/phys3810/grover.py); this notebook calls that implementation rather than maintaining a second circuit implementation.


## 1. Setup and conventions · 5 minutes

This lesson supports **2–5 qubits** and ideal local simulation. Targets are distinct, nonempty binary strings of exactly the chosen width; empty, duplicate, and all-marked sets are rejected. Iterations are explicit nonnegative integers. Returned circuits have no measurements.

In a displayed bitstring, **qubit 0 is the rightmost bit**. Thus `001` means qubit 0 is 1 and qubits 1 and 2 are 0; its amplitude-vector index is `int("001", 2) == 1`, not 4. Circuit drawings place qubit 0 on the top wire.

**Predict before running:** which amplitude-vector index corresponds to `011`? Which qubits are 1?


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

# Works from the repository root or this notebook folder, including an extracted ZIP.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "phys3810" / "grover.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Keep the extracted repository together and launch Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram
from src.phys3810.grover import (
    diffusion_operator,
    grover_circuit,
    grover_iteration,
    phase_oracle,
)

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."
print("Python:", sys.version.split()[0], "| Qiskit:", version("qiskit"))
print("Interpreter:", sys.executable)
print("Bitstring 011 has index", int("011", 2), "and sets qubits 1 and 0.")

# Analysis helpers only: expected probabilities come from the equation, not a reference circuit.
def marked_probability(circuit, targets):
    probabilities = Statevector.from_instruction(circuit).probabilities()
    return float(sum(probabilities[int(target, 2)] for target in targets))


def analytic_probability(num_qubits, num_marked, iterations):
    theta = np.arcsin(np.sqrt(num_marked / (2**num_qubits)))
    return np.sin((2 * iterations + 1) * theta) ** 2


## 2. The oracle changes phase, not the measured label · 10 minutes

Let $S$ be the marked set. The phase oracle applies

$$O|x\rangle = (-1)^{[x\in S]}|x\rangle.$$

For target `001`, only that basis state's amplitude changes sign. The oracle does not write the answer into an extra output qubit. Immediately measuring a uniformly prepared state after the oracle still gives a uniform distribution: a minus sign alone does not change a squared magnitude.

**Predict:** after marking `001`, will its probability be larger than $1/8$? What changed that a later interference step could use?


In [ ]:
targets = ("001",)
oracle = phase_oracle(3, targets)

# Entire matrix comparison checks every basis state, including off-diagonal zeros.
expected_diagonal = np.ones(8)
expected_diagonal[int("001", 2)] = -1
np.testing.assert_allclose(Operator(oracle).data, np.diag(expected_diagonal), rtol=0, atol=1e-12)

uniform = Statevector(np.ones(8, dtype=complex) / np.sqrt(8))
after_oracle = uniform.evolve(oracle)
np.testing.assert_allclose(after_oracle.data, expected_diagonal / np.sqrt(8), rtol=0, atol=1e-12)
np.testing.assert_allclose(after_oracle.probabilities(), np.full(8, 1/8), rtol=0, atol=1e-12)
print("Oracle signs in basis order 000, 001, ..., 111:", expected_diagonal.astype(int))
print("Probabilities after phase marking:", after_oracle.probabilities())
print("PASSED: only 001 changes sign; all eight probabilities remain 1/8")


## 3. Diffusion converts the phase difference into amplification · 10 minutes

The uniformly prepared state is $|s\rangle = H^{\otimes n}|0\cdots0\rangle$. We use the exact diffusion convention

$$D = 2|s\rangle\langle s| - I.$$

For an amplitude vector with mean $\bar a$, this maps each amplitude to $2\bar a-a_x$: reflection about the mean. One **Grover iteration** is $Q=DO$, so the oracle acts first and diffusion acts second. The circuit first prepares $|s\rangle$, then applies $Q$ repeatedly.

**Predict:** which amplitude becomes large after reflecting the sign-marked vector? Would measuring between the oracle and diffusion preserve this interference experiment?


In [ ]:
diffusion = diffusion_operator(3)
expected_diffusion = 2 * np.ones((8, 8)) / 8 - np.eye(8)
np.testing.assert_allclose(Operator(diffusion).data, expected_diffusion, rtol=0, atol=1e-12)

mean_amplitude = np.mean(after_oracle.data)
after_diffusion = after_oracle.evolve(diffusion)
np.testing.assert_allclose(
    after_diffusion.data, 2 * mean_amplitude - after_oracle.data, rtol=0, atol=1e-12
)
np.testing.assert_allclose(
    uniform.evolve(grover_iteration(3, targets)).data,
    after_diffusion.data, rtol=0, atol=1e-12,
)
np.testing.assert_allclose(after_diffusion.probabilities()[1], 25/32, rtol=0, atol=1e-12)
print("Marked amplitude before oracle / after oracle / after diffusion:")
print(uniform.data[1], after_oracle.data[1], after_diffusion.data[1])
print("PASSED: one iteration raises P(001) from 1/8 to 25/32")


## 4. Single-target lab: predict, execute, compare · 20 minutes

Write $N=2^n$, let $M$ count distinct marked states, and define $\theta=\arcsin\sqrt{M/N}$. For uniform preparation, after $r$ iterations the **total** probability of measuring a marked state is

$$P_r = \sin^2\!\left((2r+1)\theta\right).$$

Here $r=0$ still includes state preparation, so $P_0=M/N$. Geometrically, each iteration rotates the state by $2\theta$ in the marked/unmarked two-dimensional subspace.

**Predict:** for $N=4,M=1$, $\theta=\pi/6$. What are $P_0$, $P_1$, and $P_2$? More iterations do not always help.

Start with a two-qubit search for `01`. The circuit figure labels three blocks from left to right: **Prepare** applies Hadamard gates to make the uniform state, **Oracle O** marks the target phase, and **Diffusion D** reflects the amplitudes. It contains no measurements. These are gates made from the canonical source circuits, not a second implementation.

The diffusion block retains the module's overall phase convention internally. When expanded, a `Global Phase: π` annotation changes the phase of the whole state equally and cannot change measurement probabilities. The module tracks that phase explicitly to match the stated exact diffusion convention; the oracle's relative marked/unmarked phase still matters.


In [ ]:
small_targets = ("01",)
small = grover_circuit(2, small_targets, iterations=1)
assert small.num_clbits == 0  # Measurements belong to a separate sampling copy.
np.testing.assert_allclose(marked_probability(small, small_targets), 1.0, rtol=0, atol=1e-12)
np.testing.assert_allclose(
    marked_probability(grover_circuit(2, small_targets, iterations=0), small_targets),
    1/4, rtol=0, atol=1e-12,
)
# Group the same canonical operations into labeled teaching blocks.
teaching_circuit = QuantumCircuit(2)
teaching_circuit.append(grover_circuit(2, small_targets, iterations=0).to_gate(label="Prepare"), range(2))
teaching_circuit.append(phase_oracle(2, small_targets).to_gate(label="Oracle O"), range(2))
teaching_circuit.append(diffusion_operator(2).to_gate(label="Diffusion D"), range(2))
np.testing.assert_allclose(Operator(teaching_circuit).data, Operator(small).data, rtol=0, atol=1e-12)
figure = teaching_circuit.draw("mpl", fold=25)
display(figure)
plt.close(figure)
print("PASSED: four candidates, one marked state, one iteration gives success probability 1")


Now return to eight candidates and target `001`. Two iterations give $P_2=121/128\approx0.9453125$, not certainty. The remaining probability is spread across the seven unmarked states.

The exact statevector checks the mathematical claim. A seeded V2 sampler then produces 2,048 measurement shots. The histogram should have a large `001` bar and small bars elsewhere; random counts need not match another computer exactly.


In [ ]:
search = grover_circuit(3, targets, iterations=2)
exact = marked_probability(search, targets)
np.testing.assert_allclose(exact, 121/128, rtol=0, atol=1e-12)
np.testing.assert_allclose(exact, analytic_probability(3, 1, 2), rtol=0, atol=1e-12)
np.testing.assert_allclose(np.linalg.norm(Statevector.from_instruction(search).data), 1.0, rtol=0, atol=1e-12)

# Keep the unmeasured circuit for exact checks; add measurement to a copy for sampling.
measured = search.measure_all(inplace=False)
shots = 2048
sample = StatevectorSampler(seed=2026).run([measured], shots=shots).result()[0]
counts = sample.data.meas.get_counts()
assert search.num_clbits == 0
assert sum(counts.values()) == shots
assert set(counts) <= {format(index, "03b") for index in range(8)}
observed_marked = counts.get("001", 0)
expected_marked = shots * (121/128)
sigma = np.sqrt(shots * (121/128) * (1 - 121/128))
assert abs(observed_marked - expected_marked) <= 6 * sigma
print("Exact P(001):", exact)
print("Sample counts:", dict(sorted(counts.items())))
print("Observed marked fraction:", observed_marked / shots)

figure = plot_histogram(counts, figsize=(8, 4), title="Target 001 · two iterations · 2,048 shots")
display(figure)
plt.close(figure)
print("PASSED: exact 121/128 probability, shot accounting, and sampled agreement within six standard deviations")


## 5. Core exit questions · 5 minutes

1. Why did the oracle alone leave the probabilities unchanged?
2. What does diffusion do to amplitudes, and why does operation order matter?
3. Why did we preserve an unmeasured circuit for exact checks?
4. For the two-qubit example, predict the effect of a second iteration before running the expansion.

<details>
<summary>Check your explanations after discussing</summary>

A sign flip preserves amplitude magnitudes. Diffusion reflects amplitudes about their mean and uses relative phase to change probabilities. We apply oracle then diffusion to the uniformly prepared state. Measurements sample outcomes and disturb the coherent state, so exact simulation uses the unmeasured circuit. For four candidates and one target, a second iteration reduces success probability from 1 to 1/4.

</details>


## 6. Expansion: mark a set of answers · 15 minutes

Use targets `001` and `011`. They differ at qubit 1 and share qubit 0 equal to 1. Now $N=8,M=2$, so one iteration gives total marked probability 1, divided equally between those two answers.

**Prediction exercise:** does one marked state's probability equal $P_r$, or do we sum over all marked states? Try changing the pair while keeping $M=2$. Which result should remain unchanged?

The short input-contract exercise immediately below is included within these 15 minutes.


In [ ]:
multiple_targets = ("001", "011")
assert len(multiple_targets) == 2, "This exercise uses exactly two targets."
multiple_oracle = phase_oracle(3, multiple_targets)
# Expected locations come from the selected labels, independently of the circuit.
multiple_indices = [int(target, 2) for target in multiple_targets]
multiple_diagonal = np.ones(8)
multiple_diagonal[multiple_indices] = -1
np.testing.assert_allclose(Operator(multiple_oracle).data, np.diag(multiple_diagonal), rtol=0, atol=1e-12)

multiple_search = grover_circuit(3, multiple_targets, iterations=1)
multiple_probabilities = Statevector.from_instruction(multiple_search).probabilities()
expected_probabilities = np.zeros(8)
expected_probabilities[multiple_indices] = 1/2
np.testing.assert_allclose(multiple_probabilities, expected_probabilities, rtol=0, atol=1e-12)
np.testing.assert_allclose(marked_probability(multiple_search, multiple_targets), 1.0, rtol=0, atol=1e-12)
np.testing.assert_allclose(analytic_probability(3, 2, 1), 1.0, rtol=0, atol=1e-12)
print(f"PASSED: targets {', '.join(multiple_targets)} each have probability 1/2 after one iteration")


### Input-contract exercise · part of the 15-minute marked-set segment

Our oracle lists its marked states explicitly. Repeating a target would otherwise risk flipping its phase twice, so the module rejects duplicates instead of guessing the caller's intent. It also rejects an empty or all-marked set. Test invalid input without allowing an unhandled cell error:


In [ ]:
try:
    phase_oracle(3, ("001", "001"))
except ValueError as error:
    print("Expected input rejection:", error)
else:
    raise AssertionError("Duplicate targets should be rejected.")
print("PASSED: intentional invalid-input demonstration was handled inside the cell")


## 7. Expansion: too many iterations · 10 minutes

The figure below compares the formula (lines joining integer iteration counts) with exact circuit results (markers). For $N=4,M=1$, success is 1 at one iteration and falls to 1/4 at two. For $N=8,M=1$, the first high-probability peak is near two iterations. The oscillation is a consequence of repeated rotations, not a numerical simulation failure.

**Experiments:** predict what happens when $M$ changes; then change the target set and compute its formula before running the circuit. Keep $2\le n\le5$ and $0<M<N$. Explain why blindly increasing the iteration count is a poor strategy.


In [ ]:
# Edit the target set in either example to explore a different marked-state count.
plot_examples = [(2, ("01",), "tab:blue"), (3, ("001",), "tab:orange")]
iterations = np.arange(8)
figure, axes = plt.subplots(figsize=(8, 4.8))
for num_qubits, marked, color in plot_examples:
    predicted = analytic_probability(num_qubits, len(marked), iterations)
    simulated = np.array([
        marked_probability(grover_circuit(num_qubits, marked, iterations=int(r)), marked)
        for r in iterations
    ])
    np.testing.assert_allclose(simulated, predicted, rtol=0, atol=1e-12)
    axes.plot(iterations, predicted, "--", color=color, label=f"N={2**num_qubits}, M={len(marked)}: formula")
    axes.scatter(iterations, simulated, color=color, s=45, label=f"N={2**num_qubits}, M={len(marked)}: circuit")
np.testing.assert_allclose(
    marked_probability(grover_circuit(2, ("01",), iterations=2), ("01",)),
    1/4, rtol=0, atol=1e-12,
)
axes.set(xlabel="Grover iterations r (integer)", ylabel="Total marked-state probability", ylim=(-0.03, 1.06))
axes.set_xticks(iterations)
axes.set_title("Amplitude amplification oscillates")
axes.grid(alpha=0.25)
axes.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9)
figure.tight_layout()
display(figure)
plt.close(figure)
print("PASSED: iteration sweeps agree with the formula; a second N=4 iteration gives 1/4")


## What this lab establishes—and its limits

You checked oracle action, diffusion, exact success probabilities, measurement accounting, multiple targets, and overshooting. The repository's [independent tests](../../../tests/test_grover.py) cover more inputs and compare the iteration operator with Qiskit's `grover_operator()` up to a global phase. That SDK comparison supplements the mathematical checks rather than defining their expected answers.

This implementation uses **enumerated marked states**, **2–5 qubits**, and **ideal local simulation**. We already supply the targets in order to build an instructional oracle; this is not an end-to-end search benchmark. Preparing an oracle, synthesizing controlled operations, state preparation, and readout have costs. Counting oracle calls alone does not measure those costs, and these examples do not demonstrate quantum advantage.

Run `python scripts/validate_classroom.py` from the repository root for the full required classroom checks. It executes this notebook in a fresh kernel and saves an executed copy under `artifacts/executed-classroom/`; the source notebook stays output-free.

References: [IBM's Grover operator API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.grover_operator) explains the reference construction and phase convention; [IBM's bit-ordering guide](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering) documents qubit indices, labels, and integers. See [the course plan](../COURSE_PLAN.md) for the later QFT/QPE and HHL/Shor sessions.
